In [28]:
!uv pip install -q pandas plotly
!uv pip install -q kaleido nbformat

In [29]:
import pandas as pd
import glob
import json
import plotly.express as px

def load_data(paths):
    files = []
    for p in paths:
        files.extend(glob.glob(f"{p}/**/result.json", recursive=True))
    return pd.DataFrame([json.load(open(f)) for f in files])

def clean_model_ids(model_ids: pd.Series):
    return model_ids.str.split('/').str[-1].str.replace("Instruct", "it").str.replace("b", "B").str.replace("270m", "270M").str.replace("-it", "")

In [30]:
paths = [
    './multirun/full-head-final'
]
df = load_data(paths)
df = df.round(6)
df = df.drop(columns=['times', 'autorange_min_run_time'])
df = df.sort_values(by=['model_id'], ascending=True, key=lambda col: col.str.lower())
model_order = ['gemma-3-270M', 'gemma-3-1B', 'Qwen3-0.6B', 'Qwen3-1.7B', 'Llama-3.2-1B', 'Llama-3.2-3B']

df["model_id"] = clean_model_ids(df["model_id"])
df.head()

,k,benchmark_lm_head_only,sleep,model_id,ef,batch_size,is_ref,mean,median,p25,p75,iqr,num_threads
14,50,True,4.0,gemma-3-1B,200,1,False,0.003643,0.003561,0.003548,0.003601,0.000052,6
15,50,True,4.0,gemma-3-1B,200,1,True,0.068971,0.068874,0.068735,0.069231,0.000496,6
16,50,True,4.0,gemma-3-1B,200,2,False,0.005370,0.005165,0.005127,0.005545,0.000418,6
17,50,True,4.0,gemma-3-1B,200,2,True,0.054960,0.054695,0.054168,0.055660,0.001492,6
18,50,True,4.0,gemma-3-1B,200,4,False,0.008310,0.008058,0.007990,0.008408,0.000418,6


In [31]:
def compare_runs(df, index_cols):
    df = df.groupby([*index_cols, 'is_ref']).first().reset_index()

    ref_median = df[df['is_ref'] == True].set_index(index_cols)[['median']]
    opt_median = df[df['is_ref'] == False].set_index(index_cols)[['median']]

    speedup = ref_median / opt_median

    dfs = ref_median.join(opt_median, lsuffix='_ref', rsuffix='_nonref')
    dfs['speedup'] = speedup

    return dfs.reset_index()

dfs = compare_runs(df, index_cols=['model_id', 'ef', 'batch_size'])
dfs[dfs["batch_size"] == 1]

,model_id,ef,batch_size,median_ref,median_nonref,speedup
0,Llama-3.2-1B,200,1,0.059700,0.005344,11.171407
7,Llama-3.2-3B,200,1,0.091068,0.006526,13.954643
14,Qwen3-0.6B,200,1,0.035585,0.003397,10.475419
21,Qwen3-1.7B,200,1,0.071604,0.006767,10.581351
28,gemma-3-1B,200,1,0.068874,0.003561,19.341196
35,gemma-3-270M,200,1,0.038497,0.001670,23.052096


In [32]:
paper_style_dict = {
    'layout.plot_bgcolor': 'rgba(0, 0, 0, 0)',
    'layout.font.family': 'monospace', # 'Times New Roman',
    'layout.font.size': 20,
    'layout.xaxis.linecolor': 'black',
    'layout.xaxis.ticks': 'outside',
    'layout.xaxis.mirror': True,
    'layout.xaxis.showline': True,
    'layout.xaxis.showgrid': True,
    'layout.xaxis.gridcolor': 'lightgray',
    'layout.xaxis.gridwidth': 0.1,
    'layout.yaxis.linecolor': 'black',
    'layout.yaxis.ticks': 'outside',
    'layout.yaxis.mirror': True,
    'layout.yaxis.showline': True,
    'layout.yaxis.showgrid': True,
    'layout.yaxis.gridcolor': 'lightgray',
    'layout.yaxis.gridwidth': 0.1,
    'layout.autosize': False,
    'layout.showlegend': True,
    'layout.legend.bgcolor': 'rgba(0, 0, 0, 0)',
    'layout.legend.xanchor': 'right',
    'layout.legend.x': 0.98,
    'layout.legend.yanchor': 'top',
    'layout.legend.y': 0.98,
    'layout.legend.font.family': 'monospace',
    'layout.legend.font.size': 16,
    'layout.xaxis.gridwidth': 1, 
    'layout.yaxis.gridwidth': 1,
    'layout.margin': {'l': 20, 'r': 20, 't': 20, 'b': 20},
}

In [33]:
def plot(df, ylim_max=None):
    fig = px.line(
        df, 
        x="batch_size", 
        y="speedup",
        range_y=[0, ylim_max],
        range_x=[1, 32],
        color="model_id",
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Dark2,
        category_orders={'model_id': model_order},
        labels={
            "batch_size": "Batch Size",
            "speedup": "Speedup (×)",
            "model_id": "Model Architecture"
        }
    )

    fig.add_hline(
        y=1, 
        line_dash="dash", 
        line_color="black", 
        annotation_text="Baseline", 
        annotation_position="top left",
        line_width=1
    )


    fig.update_layout(xaxis = dict(tickvals = list(df["batch_size"])))

    fig.update(**paper_style_dict)
    return fig

#print("EF 100")
#fig = plot(dfs[(dfs["ef"] == 100)], ylim_max=35)
#fig.write_image("head-speedup-ef-100.pdf")
#fig.show()

print("EF 200")
fig = plot(dfs[(dfs["ef"] == 200)], ylim_max=25)
fig.write_image("head-speedup-ef-200.pdf")
fig.show()

EF 200


## Full Model

In [ ]:
paths = [
    # './multirun/full-sweep-final',
    # './multirun/2026-05-03/00-42-09',
]
df = load_data(paths)
df = df.round(6)
df = df.drop(columns=['times', 'autorange_min_run_time'])
df = df.sort_values(by=['model_id'], ascending=True, key=lambda col: col.str.lower())
df["model_id"] = clean_model_ids(df["model_id"])

dfs = compare_runs(df, index_cols=['model_id', 'ef', 'batch_size', 'new_token_count'])
dfs.head()

KeyError: 'new_token_count'

In [40]:
def plot(df):
    fig = px.line(
        df, 
        x="batch_size", 
        y="speedup",
        range_x=[1, 32],
        color="model_id",
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Dark2,
        category_orders={'model_id': model_order},
        labels={
            "batch_size": "Batch Size",
            "speedup": "Speedup (×)",
            "model_id": "Model Architecture"
        }
    )

    fig.add_hline(
        y=1, 
        line_dash="dash", 
        line_color="black", 
        annotation_text="Baseline", 
        annotation_position="top left",
        line_width=1
    )


    fig.update_layout(xaxis = dict(tickvals = list(df["batch_size"])))

    fig.update(**paper_style_dict)
    return fig

print("EF 200")
fig = plot(dfs)
fig.write_image("full-speedup-ef-200.pdf")
fig.show()

EF 200


## Recall

In [36]:
paths = ['./multirun/recall-final']
df = load_data(paths)
df = df.round(6)
df = df.drop(columns=["print_error"])
model_order = ['gemma-3-270M', 'gemma-3-1B', 'Qwen3-0.6B', 'Qwen3-1.7B', 'Llama-3.2-1B', 'Llama-3.2-3B']

df["model_id"] = clean_model_ids(df["model_id"])
df['model_id'] = pd.Categorical(df['model_id'], categories=model_order, ordered=True)
df = df.sort_values('model_id')

recall_cols = df.filter(like='average_top').columns
df[recall_cols] = (df[recall_cols] * 100).round(1)

recall_cols = df.filter(like='recall').columns
df[recall_cols] = (df[recall_cols] * 100).round(1)

df.to_csv("result/recall-result.csv")
df

KeyError: "['print_error'] not found in axis"

In [ ]:
df[["model_id", "average_top1_token_probability", "recall-top-1-ef-100", "recall-top-1-ef-200", "recall-top-1-ef-400"]]

,model_id,average_top1_token_probability,recall-top-1-ef-100,recall-top-1-ef-200,recall-top-1-ef-400
2,gemma-3-270M,64.8,97.7,99.2,99.8
4,gemma-3-1B,61.7,98.3,99.6,99.9
3,Qwen3-0.6B,57.4,97.7,99.4,99.9
5,Qwen3-1.7B,66.8,96.7,99.2,99.8
1,Llama-3.2-1B,48.8,98.1,99.6,99.9
0,Llama-3.2-3B,53.5,97.3,99.2,99.8


In [ ]:
df[["model_id", "average_top2_token_probability", "recall-top-2-ef-100", "recall-top-2-ef-200", "recall-top-2-ef-400"]]

,model_id,average_top2_token_probability,recall-top-2-ef-100,recall-top-2-ef-200,recall-top-2-ef-400
2,gemma-3-270M,77.3,97.3,99.1,99.7
4,gemma-3-1B,74.2,98.0,99.5,99.9
3,Qwen3-0.6B,71.1,97.5,99.4,99.9
5,Qwen3-1.7B,79.3,96.3,99.0,99.8
1,Llama-3.2-1B,62.1,97.9,99.5,99.9
0,Llama-3.2-3B,66.8,97.1,99.2,99.8
